In [0]:
payments_df = spark.read.table("global_warehouse.raw_global_warehouse.br_payments")
invoice_line_items_df = spark.read.table("global_warehouse.raw_global_warehouse.br_invoice_line_items")
invoices_df = spark.read.table("global_warehouse.raw_global_warehouse.br_invoices")
exchange_rates_df = spark.read.table("global_warehouse.raw_global_warehouse.br_exchange_rates")
products_df = spark.read.table("global_warehouse.raw_global_warehouse.br_products")
customers_df = spark.read.table("global_warehouse.raw_global_warehouse.br_customers")
regions_df = spark.read.table("global_warehouse.raw_global_warehouse.br_regions")


In [0]:
from pyspark.sql.functions import col, round

payments_with_currency = payments_df.join(
    invoices_df.select("invoice_id", "currency"), 
    "invoice_id", 
    "inner"
)

payments_with_currency = payments_with_currency.withColumnRenamed("currency", "original_currency")

payments_usd = payments_with_currency.join(
    exchange_rates_df,
    (col("original_currency") == exchange_rates_df.currency) & 
    (col("payment_date") == exchange_rates_df.date),
    "left"
)

payments_usd = payments_usd.withColumn(
    "payment_amount_usd",
    round(col("payment_amount") * col("rate_to_usd"), 2)
)

payments_usd = payments_usd.select(
    "payment_id",
    "invoice_id",
    "payment_date",
    "payment_amount",
    "original_currency", 
    "payment_amount_usd",
    "payment_method"
)

display(payments_usd)

In [0]:
from pyspark.sql.functions import col, round

products_with_invoice = products_df.join(
    invoice_line_items_df.select("product_id", "invoice_id"),
    "product_id",
    "inner"
).join(
    invoices_df.select("invoice_id", "currency", "invoice_date"),
    "invoice_id",
    "inner"
)

products_with_invoice = products_with_invoice.withColumnRenamed("currency", "original_currency")

products_with_currency = products_with_invoice.join(
    exchange_rates_df,
    (col("original_currency") == exchange_rates_df.currency) & 
    (col("invoice_date") == exchange_rates_df.date),
    "left"
)

products_usd = products_with_currency.withColumn(
    "cost_price_usd",
    round(col("cost_price") * col("rate_to_usd"), 2)
)

products_usd = products_usd.select(
    "product_id",
    "product_name",
    "cost_price",
    "original_currency",
    "cost_price_usd",
)

display(products_usd)

In [0]:
catalog_name = "global_warehouse"
schema_name = "staging_global_warehouse"
table_name = "sl_customers"
table_path = f"{catalog_name}.{schema_name}.{table_name}"

customers_df.write.mode("overwrite").saveAsTable(table_path)


In [0]:
catalog_name = "global_warehouse"
schema_name = "staging_global_warehouse"
table_name = "sl_exchange_rates"
table_path = f"{catalog_name}.{schema_name}.{table_name}"

exchange_rates_df.write.mode("overwrite").saveAsTable(table_path)

In [0]:
catalog_name = "global_warehouse"
schema_name = "staging_global_warehouse"
table_name = "sl_invoice_line_items"
table_path = f"{catalog_name}.{schema_name}.{table_name}"

invoice_line_items_df.write.mode("overwrite").saveAsTable(table_path)

In [0]:
catalog_name = "global_warehouse"
schema_name = "staging_global_warehouse"
table_name = "sl_invoices"
table_path = f"{catalog_name}.{schema_name}.{table_name}"

invoices_df.write.mode("overwrite").saveAsTable(table_path)

In [0]:
catalog_name = "global_warehouse"
schema_name = "staging_global_warehouse"
table_name = "sl_payments"
table_path = f"{catalog_name}.{schema_name}.{table_name}"

payments_usd.write.mode("overwrite").saveAsTable(table_path)

In [0]:
catalog_name = "global_warehouse"
schema_name = "staging_global_warehouse"
table_name = "sl_products"
table_path = f"{catalog_name}.{schema_name}.{table_name}"

products_usd.write.mode("overwrite").saveAsTable(table_path)

In [0]:
catalog_name = "global_warehouse"
schema_name = "staging_global_warehouse"
table_name = "sl_regions"
table_path = f"{catalog_name}.{schema_name}.{table_name}"

regions_df.write.mode("overwrite").saveAsTable(table_path)